# tm_tests_33 — Financial Tweet Sentiment Classification
## Text Mining 2025/2026 — NOVA IMS
### All Experiments with Evaluation

**Task:** Multiclass classification of financial tweets (Bearish=0, Bullish=1, Neutral=2)
**Dataset:** 9,543 train | 2,388 test
**Submitted model (single):** Fine-tuned Twitter-RoBERTa-large (2022-154m) - OOF F1 0.8859 (reproduced in tm_final_33.ipynb)
**Backbone search:** 4 transformer encoders compared - base-sentiment-latest 0.8399, base-2022-154m 0.8550, large-2022-154m **0.8859**, DeBERTa-v3 (bf16 unstable); the large Twitter model wins.
**Stacking ensemble (extra work):** 6-learner stack - OOF F1 0.8483 (now surpassed by the single large model)
**Decoder (extra work):** GPT-2 fine-tuned for classification; few-shot prompting collapses to the majority class.
**Grade targets:** 17 base pts + up to 3.50 extra pts


## 0. Setup and Imports

In [1]:
import os, sys, json, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
matplotlib.use('Agg')
sys.path.insert(0, 'src')

from src.utils import set_global_seed, CLASSES, SEED
set_global_seed()

sns.set_theme(style='whitegrid')
print(f'Python: {sys.version}')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')
print('Random seed fixed:', SEED)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
NumPy: 2.4.4, Pandas: 2.3.3
Random seed fixed: 42


## 1. Data Exploration (2.00 pts)

**Objective:** Understand the dataset before building any models.

Key findings:
- **Class imbalance:** Neutral (64.7%) >> Bullish (20.1%) > Bearish (15.1%) → imbalance ratio 4.28:1
- **Tweet length:** Mean 17 words (Twitter's 280 char limit visible)
- **Cashtags:** Bullish tweets have more cashtags (0.34 avg vs 0.22 for Bearish)
- **Consequence:** `class_weight='balanced'` in all models; 5-fold stratified CV preserves distribution


In [2]:
# Load data
train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'\nLabel distribution:')
print(train['label'].value_counts())
print(f'\nImbalance ratio: {train["label"].value_counts().max() / train["label"].value_counts().min():.2f}')
print(f'\nSample tweets:')
for label, name in CLASSES.items():
    ex = train[train['label']==label]['text'].iloc[0]
    print(f'  [{name}] {ex[:100]}')


Train: (9543, 2) | Test: (2388, 2)

Label distribution:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64

Imbalance ratio: 4.28

Sample tweets:
  [Bearish] $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
  [Bullish] $ALTG: Dougherty & Company starts at Buy
  [Neutral] $LB - MKM Partners puts a number on Victoria's Secret https://t.co/VSzHLqLBgE


In [3]:
# Class distribution plot
counts = train['label'].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(['Bearish (0)', 'Bullish (1)', 'Neutral (2)'], counts.values,
            color=['#d62728', '#2ca02c', '#1f77b4'])
axes[0].set_title('Class Distribution — Train Set')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')
axes[1].pie(counts.values, labels=['Bearish', 'Bullish', 'Neutral'],
            autopct='%1.1f%%', colors=['#d62728', '#2ca02c', '#1f77b4'])
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()


In [4]:
# Text length analysis
train['word_count'] = train['text'].str.split().str.len()
train['char_count'] = train['text'].str.len()
print(train.groupby('label')[['word_count', 'char_count']].describe().round(2))


      word_count                                          char_count         \
           count   mean   std  min  25%   50%   75%   max      count   mean   
label                                                                         
0         1442.0  12.00  4.31  3.0  9.0  11.0  15.0  32.0     1442.0  83.32   
1         1923.0  11.93  4.32  2.0  9.0  11.0  14.0  29.0     1923.0  80.38   
2         6178.0  12.30  4.84  1.0  9.0  12.0  16.0  29.0     6178.0  88.09   

                                                
         std   min    25%   50%     75%    max  
label                                           
0      32.98  20.0  57.25  75.5  110.75  151.0  
1      32.32   9.0  57.00  74.0  103.00  154.0  
2      36.17   2.0  59.00  82.0  125.00  190.0  


In [5]:
# Twitter-specific features
def extract_twitter_features(text):
    return {
        'hashtags': len(re.findall(r'#\w+', str(text))),
        'mentions': len(re.findall(r'@\w+', str(text))),
        'cashtags': len(re.findall(r'\$[A-Z]{1,5}', str(text))),
        'urls': len(re.findall(r'https?://\S+', str(text))),
    }

tf = pd.DataFrame([extract_twitter_features(t) for t in train['text']])
train_eda = pd.concat([train.reset_index(drop=True), tf], axis=1)
print('Mean Twitter features by class:')
print(train_eda.groupby('label')[['hashtags', 'mentions', 'cashtags', 'urls']].mean().round(3))


Mean Twitter features by class:
       hashtags  mentions  cashtags   urls
label                                     
0         0.165     0.021     0.216  0.510
1         0.158     0.017     0.340  0.438
2         0.265     0.046     0.171  0.569


In [6]:
# Word clouds by class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (label, name) in zip(axes, CLASSES.items()):
    text = ' '.join(train[train['label'] == label]['text'])
    cmap = 'RdYlGn' if label == 1 else ('Reds' if label == 0 else 'Blues')
    wc = WordCloud(width=400, height=200, background_color='white',
                   colormap=cmap, max_words=80).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(f'{name}')
    ax.axis('off')
plt.suptitle('Word Clouds by Sentiment Class', fontsize=14)
plt.tight_layout()
plt.show()


In [7]:
# Top TF-IDF terms per class (on preprocessed data)
from sklearn.feature_extraction.text import TfidfVectorizer
for label, name in CLASSES.items():
    subset = train[train['label'] == label]['text']
    vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
    tfidf = vec.fit_transform(subset)
    scores = sorted(zip(vec.get_feature_names_out(), tfidf.sum(axis=0).A1),
                    key=lambda x: x[1], reverse=True)[:10]
    print(f'\n{name} — Top 10 TF-IDF terms:')
    for term, s in scores:
        print(f'  {term}: {s:.2f}')



Bearish — Top 10 TF-IDF terms:
  co: 65.22
  https: 65.09
  https co: 65.09
  to: 44.50
  the: 38.53
  in: 36.42
  on: 33.94
  of: 32.19
  after: 29.04
  as: 26.51

Bullish — Top 10 TF-IDF terms:
  co: 79.34
  https: 78.62
  https co: 78.62
  to: 58.15
  on: 52.94
  in: 46.31
  the: 45.10
  up: 43.15
  stock: 39.43
  beats: 34.77

Neutral — Top 10 TF-IDF terms:
  co: 318.18
  https: 315.32
  https co: 315.32
  the: 215.84
  to: 211.18
  of: 166.99
  in: 156.23
  for: 138.82
  on: 121.02
  and: 115.50


## 2. Data Preprocessing (3.00 pts)

**Techniques implemented (≥4):**
1. **Twitter noise cleaning** (regex): URLs→`URL`, mentions→`USER`, cashtags→`TICKER_X`, hashtags→text
2. **Unicode normalization** (NFKD + ASCII): removes diacritics and special characters
3. **Stop words removal** (NLTK): preserves negations (not, no, never) — Manning et al. (2008)
4. **WordNet Lemmatization** (NLTK): reduces words to valid dictionary forms
5. **Snowball Stemming** (NLTK): more aggressive morphological reduction
6. **TweetTokenizer** (NLTK): handles emoticons, reduces character repetitions

**Ablation study result:** Raw text (no preprocessing) achieves highest F1-macro (0.7183).
This is typical in financial NLP: ticker symbols ($AAPL), analyst names, and financial terms
carry strong predictive signal that preprocessing would destroy (Go et al., 2009).


In [8]:
# Preprocessing functions
from src.preprocessing import (clean_twitter_noise, normalize_text,
                                 remove_stopwords, lemmatize_tokens,
                                 stem_tokens, tokenize_tweet, full_pipeline)

# Demonstrate each technique
sample = '$AAPL - Apple beats earnings estimates, JPMorgan raises target https://t.co/abc123 @analyst'
print('Original:', sample)
print('1. Clean Twitter noise:', clean_twitter_noise(sample))
print('2. Normalize:', normalize_text(clean_twitter_noise(sample)))
tokens = tokenize_tweet(normalize_text(clean_twitter_noise(sample)))
print('3. Tokenize (TweetTokenizer):', tokens[:10])
print('4. Remove stopwords:', remove_stopwords(tokens))
print('5. Lemmatize:', lemmatize_tokens(remove_stopwords(tokens)))
print('6. Stem:', stem_tokens(remove_stopwords(tokens)))


Original: $AAPL - Apple beats earnings estimates, JPMorgan raises target https://t.co/abc123 @analyst
1. Clean Twitter noise: TICKER_AAPL - Apple beats earnings estimates, JPMorgan raises target URL USER
2. Normalize: ticker_aapl - apple beats earnings estimates, jpmorgan raises target url user
3. Tokenize (TweetTokenizer): ['ticker_aapl', '-', 'apple', 'beats', 'earnings', 'estimates', ',', 'jpmorgan', 'raises', 'target']
4. Remove stopwords: ['ticker_aapl', '-', 'apple', 'beats', 'earnings', 'estimates', ',', 'jpmorgan', 'raises', 'target', 'url', 'user']


5. Lemmatize: ['ticker_aapl', '-', 'apple', 'beat', 'earnings', 'estimate', ',', 'jpmorgan', 'raise', 'target', 'url', 'user']
6. Stem: ['ticker_aapl', '-', 'appl', 'beat', 'earn', 'estim', ',', 'jpmorgan', 'rais', 'target', 'url', 'user']


In [9]:
# Ablation study
ablation_df = pd.read_csv('results/tables/preprocessing_ablation.csv')
print('Preprocessing Ablation Results (5-fold CV, LR + TF-IDF 10k):')
print(ablation_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.barh(ablation_df['Config'][::-1], ablation_df['F1-macro'][::-1],
               xerr=ablation_df['Std'][::-1], capsize=4, color='#1f77b4', alpha=0.85)
ax.set_xlabel('F1-macro (5-fold CV)')
ax.set_title('Preprocessing Ablation Study')
for bar, val in zip(bars, ablation_df['F1-macro'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print('\nBest config: raw (no preprocessing) — financial keywords are informative features')


Preprocessing Ablation Results (5-fold CV, LR + TF-IDF 10k):
                           Config  F1-macro      Std
           raw (no preprocessing)  0.718339 0.005346
     clean + stopwords + stemming  0.711845 0.008868
clean + stopwords + lemmatization  0.708559 0.004234
                       clean only  0.702424 0.011295

Best config: raw (no preprocessing) — financial keywords are informative features


## 3. Feature Engineering (5.50 pts base + 1.00 extra)

### A) Bag-of-Words and TF-IDF (sparse representations)
Following the professor's Lab 1 and Lab Extra approach (CountVectorizer, TfidfVectorizer).

### B) Word2Vec Embeddings
- Skip-gram (sg=1): predicts context words from centre word (Mikolov et al., 2013)
- CBOW (sg=0): predicts centre word from context
- GloVe-Twitter-100: pre-trained on 2 billion tweets (Pennington et al., 2014)
- Mean pooling for document representation (Lab 2 approach)

### C) Transformer Encoders (obligatory + extras)
- **FinBERT** (Araci, 2019): BERT fine-tuned on financial text — obligatory
- **SBERT all-mpnet-base-v2** (Reimers & Gurevych, 2019) — **+0.50 pts extra**
- **Twitter-RoBERTa** (Barbieri et al., 2020) — **+0.50 pts extra**

### D) Financial Hand-crafted Features
VADER sentiment scores + Twitter surface stats + readability (Hutto & Gilbert, 2014).


In [10]:
# Load precomputed features
import numpy as np
X_finbert_train  = np.load('data/processed/X_finbert_train.npy')
X_sbert_train    = np.load('data/processed/X_sbert_train.npy')
X_roberta_train  = np.load('data/processed/X_roberta_train.npy')
X_glove_train    = np.load('data/processed/X_glove_train.npy')
X_w2v_sg_train   = np.load('data/processed/X_w2v_sg_train.npy')
X_w2v_cbow_train = np.load('data/processed/X_w2v_cbow_train.npy')
X_fin_train      = np.load('data/processed/X_fin_train.npy')

print('Feature representations:')
shapes = {
    'BoW (binary CountVectorizer)':    'sparse',
    'TF-IDF unigrams':                 'sparse',
    'TF-IDF unigrams+bigrams':         'sparse',
    'TF-IDF char 3-5grams':            'sparse',
    'Word2Vec Skip-gram (200d)':       str(X_w2v_sg_train.shape),
    'Word2Vec CBOW (200d)':            str(X_w2v_cbow_train.shape),
    'GloVe-Twitter (100d)':            str(X_glove_train.shape),
    'Financial features (VADER+)':     str(X_fin_train.shape),
    'FinBERT CLS (768d)':              str(X_finbert_train.shape),
    'SBERT all-mpnet (768d) [EXTRA]':  str(X_sbert_train.shape),
    'Twitter-RoBERTa (768d) [EXTRA]':  str(X_roberta_train.shape),
}
for name, shape in shapes.items():
    print(f'  {name:<42} {shape}')


Feature representations:
  BoW (binary CountVectorizer)               sparse
  TF-IDF unigrams                            sparse
  TF-IDF unigrams+bigrams                    sparse
  TF-IDF char 3-5grams                       sparse
  Word2Vec Skip-gram (200d)                  (9543, 200)
  Word2Vec CBOW (200d)                       (9543, 200)
  GloVe-Twitter (100d)                       (9543, 100)
  Financial features (VADER+)                (9543, 14)
  FinBERT CLS (768d)                         (9543, 768)
  SBERT all-mpnet (768d) [EXTRA]             (9543, 768)
  Twitter-RoBERTa (768d) [EXTRA]             (9543, 768)


In [11]:
# Build TF-IDF representations
from src.features import build_bow, build_tfidf_1g, build_tfidf_2g, build_tfidf_char

train_texts = train['text'].tolist()
test_texts  = pd.read_csv('data/raw/test.csv')['text'].tolist()

bow_vec, X_bow_train, X_bow_test = build_bow(train_texts, test_texts)
tfidf_1g_vec, X_tfidf_1g_train, X_tfidf_1g_test = build_tfidf_1g(train_texts, test_texts)
tfidf_2g_vec, X_tfidf_2g_train, X_tfidf_2g_test = build_tfidf_2g(train_texts, test_texts)
tfidf_char_vec, X_tfidf_char_train, X_tfidf_char_test = build_tfidf_char(train_texts, test_texts)

print(f'BoW:          {X_bow_train.shape}')
print(f'TF-IDF 1g:    {X_tfidf_1g_train.shape}')
print(f'TF-IDF 2g:    {X_tfidf_2g_train.shape}')
print(f'TF-IDF char:  {X_tfidf_char_train.shape}')


BoW:          (9543, 7302)
TF-IDF 1g:    (9543, 7302)
TF-IDF 2g:    (9543, 18902)
TF-IDF char:  (9543, 30000)


In [12]:
# Word2Vec exploration
from src.features import train_word2vec

print('Training Word2Vec Skip-gram (vector_size=200, window=5, epochs=20)...')
w2v_sg = train_word2vec(train_texts, vector_size=200, window=5, sg=1)

# Most similar words for financial terms
for word in ['bullish', 'bearish', 'earnings', 'downgrade', 'rally']:
    if word in w2v_sg.wv:
        similar = w2v_sg.wv.most_similar(word, topn=5)
        print(f"  '{word}': {[w for w, _ in similar]}")


Training Word2Vec Skip-gram (vector_size=200, window=5, epochs=20)...


  'bullish': ['bearish', 'obliterated', 'turns', 'swings', 'premarket;']
  'bearish': ['premarket,', 'premarket;', 'tumbling', 'heads', 'price,']
  'earnings': ['view,', 'topper', 'beat', 'comps', 'ZNGA']
  'downgrade': ['Gannett', 'Pinduoduo', '$150M', 'BAML', "Intelsat's"]
  'rally': ['edge', 'changed,', 'lower,', 'firmly', 'pull']


In [13]:
# GloVe Twitter exploration
from src.features import load_glove_twitter
glove = load_glove_twitter(dim=100)
print('GloVe Twitter-100 analogies:')
result = glove.most_similar(positive=['bullish', 'stock'], negative=['bearish'], topn=3)
print(f'  bullish + stock - bearish = {[w for w, _ in result]}')

for word in ['bull', 'bear', 'earnings']:
    if word in glove:
        print(f"  Most similar to '{word}': {[w for w, _ in glove.most_similar(word, topn=3)]}")


GloVe Twitter-100 analogies:
  bullish + stock - bearish = ['stocks', 'price', 'market']
  Most similar to 'bull': ['redbull', 'red', 'horse']
  Most similar to 'bear': ['teddy', 'cat', 'dog']
  Most similar to 'earnings': ['quarterly', 'estimates', 'profits']


In [14]:
# VADER financial features example
from src.features import extract_financial_features

examples = [
    '$AAPL beats earnings by 15%, revenue up 8% YoY — very bullish!',
    '$TSLA crashes 12%, multiple analysts downgrade to sell',
    'Market volume remains stable, no significant moves today',
]
for ex in examples:
    feats = extract_financial_features(ex)
    label = 'Bullish' if feats['vader_compound'] > 0.05 else ('Bearish' if feats['vader_compound'] < -0.05 else 'Neutral')
    print(f'  [{label}] compound={feats["vader_compound"]:.3f} cashtags={feats["n_cashtags"]}: {ex[:80]}')


  [Neutral] compound=0.000 cashtags=1: $AAPL beats earnings by 15%, revenue up 8% YoY — very bullish!
  [Neutral] compound=0.000 cashtags=1: $TSLA crashes 12%, multiple analysts downgrade to sell
  [Bullish] compound=0.155 cashtags=0: Market volume remains stable, no significant moves today


## 4. Classification Models (4.50 pts + 2.00 extra)

### Traditional ML (≥2 variations each)
- **Naïve Bayes**: MultinomialNB + ComplementNB (better for imbalanced)
- **Logistic Regression**: C=0.1, 1.0, 10.0 with class_weight='balanced'
- **Linear SVM**: LinearSVC C=0.1, 1.0 (wrapped with CalibratedClassifierCV)
- **KNN**: k=5 and k=15, cosine metric (Lab 1/2 approach)
- **MLP**: (256,128) and (512,256,128) architectures
- **Random Forest**: 100 and 300 trees, class_weight='balanced'
- **XGBoost**: 100 and 300 estimators
- **LightGBM**: 100 and 300 estimators (Optuna-tuned best model)

### Transformer Encoder for Classification (obligatory) — the key lever
- **Twitter-RoBERTa fine-tuned end-to-end**: all ~125M params, 4 epochs, lr=2e-5, batch=32,
  max_len=96, AdamW + linear warmup, class-weighted loss, bf16 on GPU, 5-fold OOF + test-prob averaging
- Lifts macro-F1 from the ~0.80 frozen-embedding plateau to **0.8399** (see Section 4b)

### Decoder Model (+1.00 pt extra)
- **GPT-2 few-shot**: prompt engineering for zero-shot/few-shot classification (Brown et al., 2020)


In [15]:
# Load all results
results_df = pd.read_csv('results/tables/full_comparison_final.csv', index_col=0)
print(f'Total models evaluated: {len(results_df)}')
print()
cols = ['Model', 'F1-macro', 'F1-macro_std', 'Accuracy', 'Precision-macro', 'Recall-macro']
print(results_df[cols].head(20).to_string())


Total models evaluated: 80

                                                 Model  F1-macro  F1-macro_std  Accuracy  Precision-macro  Recall-macro
1           Twitter-RoBERTa-large-2022-154m fine-tuned  0.885885      0.006586  0.907995         0.875442      0.897477
2                    DeBERTa-v3-base fine-tuned (fp32)  0.865261      0.006474  0.889972         0.850558      0.882231
3            Twitter-RoBERTa-base-2022-154m fine-tuned  0.855024      0.006944  0.880960         0.835124      0.880001
4                   Stacking-6 (LR meta) | +fine-tuned  0.848290      0.007290  0.877083         0.827052      0.875730
5                       Soft-Average | 6 base learners  0.841154      0.000000  0.880017         0.856555      0.827560
6   Twitter-RoBERTa-base fine-tuned (sentiment-latest)  0.839918      0.006204  0.869119         0.818752      0.867144
7                   Stacking-5 (LR meta) | frozen only  0.826878      0.008620  0.863879         0.812239      0.844690
8           

In [16]:
# Results bar chart — top 20 models
fig, ax = plt.subplots(figsize=(10, 8))
top20 = results_df.head(20)
colors = ['#2ca02c' if 'Tuned' in m else '#1f77b4' for m in top20['Model']]
ax.barh(range(len(top20)), top20['F1-macro'][::-1].values,
        xerr=top20['F1-macro_std'][::-1].values, capsize=3,
        color=colors[::-1], alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20['Model'][::-1].values, fontsize=8)
ax.set_xlabel('F1-macro (5-fold stratified CV)')
ax.set_title('Top 20 Models — F1-macro Performance')
plt.tight_layout()
plt.show()


In [17]:
# Evaluate traditional models on different feature sets
from src.evaluation import evaluate_model, CV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier

y = train['label'].values

# Best sparse model
best_lr = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
r = evaluate_model(best_lr, X_tfidf_2g_train, y, 'LR C=1.0 | TF-IDF 2g')
print(f'\nLR TF-IDF 2g F1-macro: {r["F1-macro"]:.4f}')



LR C=1.0 | TF-IDF 2g
F1-macro:  0.7325 ± 0.0040
Accuracy:  0.8012
Precision: 0.7308 | Recall: 0.7348
Overfit gap: 0.1994 | Time: 4.1s

LR TF-IDF 2g F1-macro: 0.7325


In [18]:
# Best dense model
lgbm_best = LGBMClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
r_lgbm = evaluate_model(lgbm_best, X_sbert_train, y, 'LightGBM 300 | SBERT')
print(f'\nLightGBM SBERT F1-macro: {r_lgbm["F1-macro"]:.4f}')



LightGBM 300 | SBERT
F1-macro:  0.7892 ± 0.0044
Accuracy:  0.8447
Precision: 0.8175 | Recall: 0.7670
Overfit gap: 0.2108 | Time: 38.8s

LightGBM SBERT F1-macro: 0.7892


In [19]:
# Confusion matrix
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(lgbm_best, X_sbert_train, y, cv=cv5, n_jobs=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_cv, ax=axes[0],
    display_labels=['Bearish', 'Bullish', 'Neutral'],
    cmap='Blues', normalize='true')
axes[0].set_title('Normalized Confusion Matrix')
ConfusionMatrixDisplay.from_predictions(
    y, y_pred_cv, ax=axes[1],
    display_labels=['Bearish', 'Bullish', 'Neutral'],
    cmap='Blues', normalize=None)
axes[1].set_title('Absolute Confusion Matrix')
plt.tight_layout()
plt.show()


In [20]:
# FinBERT zero-shot inference (frozen domain baseline — used only as a CLS-embedding feature)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = 'ProsusAI/finbert'
tokenizer = AutoTokenizer.from_pretrained(model_name)

sample_text = '$TSLA - Tesla Q4 earnings miss estimates, shares drop 8%'
inputs = tokenizer(sample_text, return_tensors='pt', truncation=True, max_length=128)

finbert_clf = AutoModelForSequenceClassification.from_pretrained(model_name)
finbert_clf.eval()
with torch.no_grad():
    logits = finbert_clf(**inputs).logits
probs = torch.softmax(logits, dim=-1)[0].tolist()
labels = finbert_clf.config.id2label
print('FinBERT zero-shot prediction:')
for i, p in enumerate(probs):
    print(f'  {labels[i]}: {p:.4f}')
print(f'\nPredicted: {labels[probs.index(max(probs))]}')
print()
print('NOTE: the obligatory transformer-encoder *fine-tuning* is performed end-to-end on')
print('Twitter-RoBERTa in Section 4b (script: phase7_finetune_gpu.py).')


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

FinBERT zero-shot prediction:
  positive: 0.0078
  negative: 0.9726
  neutral: 0.0196

Predicted: negative

NOTE: the obligatory transformer-encoder *fine-tuning* is performed end-to-end on
Twitter-RoBERTa in Section 4b (script: phase7_finetune_gpu.py).


## 4b. Transformer Fine-tuning + Stacking Ensemble (the decisive levers)

Phases 1-5 used the transformer encoders only as **frozen** CLS embeddings, which plateau
around macro-F1 0.80. The two improvements that break past this:

1. **End-to-end fine-tuning of Twitter-RoBERTa** (`phase7_finetune_gpu.py`): all ~125M
   parameters trained on the financial labels — 4 epochs, lr 2e-5, batch 32, max_len 96,
   AdamW + 10% linear warmup, gradient clipping 1.0, inverse-frequency class-weighted
   cross-entropy, bf16 mixed precision on GPU. Evaluated with the SAME 5-fold stratified
   protocol → out-of-fold (OOF) macro-F1, and the 5 folds' test probabilities are averaged.

2. **Out-of-fold stacking** (`phase8_final_ensemble.py`): 6 base learners (tuned LightGBM on
   RoBERTa, MLP on SBERT, XGBoost on RoBERTa, LightGBM on FinBERT, LR on TF-IDF bigrams, and
   the fine-tuned Twitter-RoBERTa) → 18-dim meta-features → Logistic Regression meta-learner.
   Identical fold seed ⇒ leakage-free, comparable OOF score.


**Backbone search (this submission).** We compared four Transformer encoders under the identical 5-fold protocol (model selection via the Hugging Face Hub) and submitted the best *single* model: `twitter-roberta-large-2022-154m` at OOF macro-F1 **0.8859** (see next cell).

In [21]:
# Fine-tuned Twitter-RoBERTa results (Phase 7, 5-fold OOF)
import json
with open('results/tables/phase7_summary.json') as f:
    ph7 = json.load(f)
print(f"Model: {ph7['Model']}")
print(f"OOF F1-macro:  {ph7['F1-macro']:.4f} +/- {ph7['F1-macro_std']:.4f}")
print(f"Accuracy:      {ph7['Accuracy']:.4f}")
print(f"Precision:     {ph7['Precision-macro']:.4f} | Recall: {ph7['Recall-macro']:.4f}")
print(f"Per-fold F1:   {[round(x,4) for x in ph7['per_fold_f1']]}")
print(f"Train time:    {ph7['elapsed_min']:.1f} min on GPU")


Model: Twitter-RoBERTa fine-tuned (full, 4ep, cuda)
OOF F1-macro:  0.8399 +/- 0.0062
Accuracy:      0.8691
Precision:     0.8188 | Recall: 0.8671
Per-fold F1:   [0.8443, 0.8282, 0.845, 0.839, 0.8428]
Train time:    4.1 min on GPU


In [22]:
# Transformer BACKBONE SEARCH - 4 encoders, identical 5-fold OOF protocol
# (Hugging Face model selection; trained only on train.csv)
import json
rows = [('phase7_summary', 'twitter-roberta-base-sentiment-latest'),
        ('phase10_roberta2022', 'twitter-roberta-base-2022-154m'),
        ('phase10_robertalarge', 'twitter-roberta-large-2022-154m  <- SUBMITTED'),
        ('phase10_debertav3', 'deberta-v3-base (fp32; bf16 was NaN-unstable)')]
print(f"{'Backbone':52s} OOF-F1   Acc")
for tag, name in rows:
    d = json.load(open(f'results/tables/{tag}.json'))
    print(f"{name:52s} {d['F1-macro']:.4f}  {d.get('Accuracy', 0):.4f}")
print('\nThe larger, domain-matched Twitter-RoBERTa wins (+0.046 over the base model);')
print('it also surpasses the 6-learner stack (0.8483), so the single model is best overall.')

Backbone                                             OOF-F1   Acc
twitter-roberta-base-sentiment-latest                0.8399  0.8691
twitter-roberta-base-2022-154m                       0.8550  0.8810
twitter-roberta-large-2022-154m  <- SUBMITTED        0.8859  0.9080
deberta-v3-base (fp32; bf16 was NaN-unstable)        0.8653  0.8900

The larger, domain-matched Twitter-RoBERTa wins (+0.046 over the base model);
it also surpasses the 6-learner stack (0.8483), so the single model is best overall.


In [23]:
# Final ensemble candidates (Phase 8) — all on identical 5-fold OOF macro-F1
import os, json, pandas as pd
if os.path.exists('results/tables/phase8_final_ensemble.csv'):
    ph8_df = pd.read_csv('results/tables/phase8_final_ensemble.csv')
    print(ph8_df[['Model', 'F1-macro', 'Accuracy', 'Precision-macro', 'Recall-macro']].to_string(index=False))
if os.path.exists('results/tables/phase8_summary.json'):
    with open('results/tables/phase8_summary.json') as f:
        ph8 = json.load(f)
    print(f"\nBest experiment (extra work): {ph8['winner']}  (OOF F1-macro={ph8['winner_f1_macro']:.4f})")
    print(f"Improvement over 0.80 baseline: {ph8['improvement_over_baseline']:+.4f}")

# The SUBMITTED solution must be a SINGLE classification model (Project Guidelines 5.2),
# so pred_33.csv is the single fine-tuned Twitter-RoBERTa - NOT the stacking ensemble.
if os.path.exists('results/tables/final_summary.json'):
    with open('results/tables/final_summary.json') as f:
        fs = json.load(f)
    print(f"\nSUBMITTED single model: {fs['final_best_model']}  (OOF F1-macro={fs['cv_f1_macro']:.4f})")
    print(f"Submission source: {fs['submission_source']} -> pred_33.csv")
    print('(The 6-learner stack scores higher but is reported as extra work, not submitted.)')


                             Model  F1-macro  Accuracy  Precision-macro  Recall-macro
Stacking-6 (LR meta) | +fine-tuned  0.848290  0.877083         0.827052      0.875730
    Soft-Average | 6 base learners  0.841154  0.880017         0.856555      0.827560
   Fine-tuned RoBERTa (standalone)  0.839918  0.869119         0.818752      0.867144
Stacking-5 (LR meta) | frozen only  0.826878  0.863879         0.812239      0.844690

Best experiment (extra work): Stacking-6 (LR meta) | +fine-tuned  (OOF F1-macro=0.8483)
Improvement over 0.80 baseline: +0.0462

SUBMITTED single model: Twitter-RoBERTa-large-2022-154m fine-tuned  (OOF F1-macro=0.8859)
Submission source: results/predictions/pred_robertalarge.csv -> pred_33.csv
(The 6-learner stack scores higher but is reported as extra work, not submitted.)


In [24]:
# GPT-2 Decoder classifier (Extra +1.00 pt)
# Cached result
import json, os
if os.path.exists('results/tables/gpt2_decoder_result.json'):
    with open('results/tables/gpt2_decoder_result.json') as f:
        gpt2 = json.load(f)
    print(f'GPT-2 decoder F1-macro: {gpt2["F1-macro"]:.4f}')
    print(f'Note: {gpt2.get("note", "")}')
else:
    print('GPT-2 result not cached. Run phase4_models.py to generate.')


GPT-2 decoder F1-macro: 0.2551
Note: 150 val samples


## 5. Hyperparameter Tuning (Optuna Bayesian Optimisation)

Used **Optuna** with TPE Sampler and Median Pruner over 50 trials.
Optimizes LightGBM hyperparameters: n_estimators, learning_rate, num_leaves, max_depth,
min_child_samples, subsample, colsample_bytree, reg_alpha, reg_lambda.


In [25]:
# Load Optuna results
with open('results/tables/optuna_best_params.json') as f:
    optuna_data = json.load(f)

print(f'Best LightGBM F1-macro: {optuna_data["best_value"]:.4f}')
print('Best parameters:')
for k, v in optuna_data['best_params'].items():
    print(f'  {k}: {v}')


Best LightGBM F1-macro: 0.8021
Best parameters:
  n_estimators: 678
  learning_rate: 0.08883265057539667
  num_leaves: 36
  max_depth: 12
  min_child_samples: 49
  subsample: 0.7517287534268207
  colsample_bytree: 0.8653122384725886
  reg_alpha: 0.003043263840972833
  reg_lambda: 0.009164056214148394


## 6. Evaluation and Results (1.50 pts)

### Metrics (required): Recall, Precision, Accuracy, F1-Score

### Error Analysis — Contextual Interpretation
The most common errors are **Neutral misclassified as Bearish/Bullish** and vice versa,
reflecting the inherent ambiguity of neutral financial text. Twitter's informal register
and heavy use of jargon make sentiment boundaries fuzzy. The model performs best on
Bullish (strong positive signal from words like "beats", "upgraded", "bullish") and
worst on Neutral where language is most ambiguous.


In [26]:
# Full classification report
from sklearn.metrics import classification_report
final_best = pd.read_csv('results/tables/full_comparison_final.csv', index_col=0).iloc[0]
print(f'Best experiment (top of comparison table): {final_best["Model"]}')
print(f'F1-macro:   {final_best["F1-macro"]:.4f} +/- {final_best["F1-macro_std"]:.4f}')
print(f'Accuracy:   {final_best["Accuracy"]:.4f}')
print(f'Precision:  {final_best["Precision-macro"]:.4f}')
print(f'Recall:     {final_best["Recall-macro"]:.4f}')
print()
print('Error analysis:')
err_df = pd.read_csv('results/tables/error_examples.csv')
print(f'Total errors: {len(err_df)} / 9543 ({100*len(err_df)/9543:.1f}%)')


Best experiment (top of comparison table): Twitter-RoBERTa-large-2022-154m fine-tuned
F1-macro:   0.8859 +/- 0.0066
Accuracy:   0.9080
Precision:  0.8754
Recall:     0.8975

Error analysis:
Total errors: 1401 / 9543 (14.7%)


In [27]:
# Display confusion matrix
from IPython.display import Image
import os
if os.path.exists('results/figures/confusion_matrix.pdf'):
    print('Confusion matrix saved at results/figures/confusion_matrix.pdf')
    # For notebook display, show the PNG version
    from sklearn.model_selection import cross_val_predict, StratifiedKFold
    from sklearn.metrics import ConfusionMatrixDisplay
    from lightgbm import LGBMClassifier
    import json

    with open('results/tables/optuna_best_params.json') as f:
        best_params = json.load(f)['best_params']

    X_sbert = np.load('data/processed/X_sbert_train.npy')
    y = train['label'].values
    best_m = LGBMClassifier(**best_params, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    y_pred = cross_val_predict(best_m, X_sbert, y, cv=cv5, n_jobs=-1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[0],
        display_labels=['Bearish', 'Bullish', 'Neutral'], cmap='Blues', normalize='true')
    axes[0].set_title('Normalized')
    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[1],
        display_labels=['Bearish', 'Bullish', 'Neutral'], cmap='Blues')
    axes[1].set_title('Absolute')
    plt.tight_layout()
    plt.show()

    print(classification_report(y, y_pred, target_names=['Bearish', 'Bullish', 'Neutral']))


Confusion matrix saved at results/figures/confusion_matrix.pdf


              precision    recall  f1-score   support

     Bearish       0.79      0.66      0.72      1442
     Bullish       0.82      0.72      0.76      1923
     Neutral       0.87      0.93      0.90      6178

    accuracy                           0.85      9543
   macro avg       0.82      0.77      0.79      9543
weighted avg       0.84      0.85      0.84      9543



In [28]:
# Error examples by type
err_df = pd.read_csv('results/tables/error_examples.csv')
label_names = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}
error_counts = err_df.groupby(['label', 'predicted']).size().sort_values(ascending=False)
print('Most frequent error types:')
print(error_counts.head(8))

print()
for (true_l, pred_l), count in error_counts.head(4).items():
    subset = err_df[(err_df['label']==true_l) & (err_df['predicted']==pred_l)]
    if len(subset) > 0:
        print(f'\nTrue={label_names[true_l]}, Predicted={label_names[pred_l]} ({count} cases):')
        for _, row in subset.head(2).iterrows():
            print(f'  -> {row["text"][:110]}...')


Most frequent error types:
label  predicted
1      2            404
0      2            357
2      1            236
       0            228
1      0            106
0      1             70
dtype: int64


True=Bullish, Predicted=Neutral (404 cases):
  -> $GBT: Cantor Fitzgerald resumes at Overweight...
  -> $KIM: Compass Point ups to Neutral...

True=Bearish, Predicted=Neutral (357 cases):
  -> $FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB...
  -> $LK - Muddy Waters goes short Luckin Coffee https://t.co/8yrbwAjLKG...

True=Neutral, Predicted=Bullish (236 cases):
  -> $WING - Baird returns to Wingstop bull camp https://t.co/KfPaweOVgo...
  -> BRP Group started at hold with $18 stock price target at Jefferies...

True=Neutral, Predicted=Bearish (228 cases):
  -> BTIG resumes coverage on range of healthcare stocks, neutral on AbbVie...
  -> Upgrades 2/4: $AMAL $CM $CMCSA $ELF $GDOT $LFUS $MLI $RRR $TD $TXRH $VLY $WW Downgrades 2/4: $ABG $AWK $CHKP $...


## 7. Extra Challenge 2: Agentic Workflow (+1.50 pts)

**See `08_Agentic_Workflow.ipynb` for the full implementation.**

An LLM-powered agent receives financial tweets via conversational interface and:
1. Routes to the appropriate classifier (VADER for quick baseline, LightGBM for accuracy)
2. Compares predictions from multiple models
3. Explains the reasoning behind the final verdict

Architecture: LangChain ReAct agent with 3 tools (VADER, LightGBM+SBERT, FinBERT),
ConversationBufferMemory for context, and a structured explanation pipeline.


## 8. Final Predictions

In [29]:
# Verify pred_33.csv
pred = pd.read_csv('pred_33.csv')
print(f'Predictions shape: {pred.shape}')
print(f'Columns: {pred.columns.tolist()}')
print(f'\nDistribution:')
print(pred['label'].value_counts())
print(f'\nFirst 5:')
print(pred.head())
assert len(pred) == 2388, 'Wrong number of predictions!'
assert pred['label'].nunique() == 3, 'Missing classes!'
print('\nAll assertions PASSED.')


Predictions shape: (2388, 2)
Columns: ['id', 'label']

Distribution:
label
2    1497
1     516
0     375
Name: count, dtype: int64

First 5:
   id  label
0   0      1
1   1      2
2   2      2
3   3      1
4   4      2

All assertions PASSED.
